# Complex Content-Based Car Recommendation System

In this case I decided to use content-based recommendation system because we don't have users, marks or car selection history, so collaborative filtering will not solve this problem. It this case we need to compare cars by their characteristics, so this method suits this task the best.

We separated the features into numerical and categorical groups. Numerical features such as horsepower, engine size, price, fuel consumption and weight are scaled and compared using distance-based similarity. Categorical features such as fuel type, body style, drive wheels and engine type are encoded using one-hot encoding and compared using cosine similarity.

The final recommendation score is a weighted combination of numerical similarity, categorical similarity and brand similarity. Numerical features receive the highest weight because technical specifications are the most important for comparing cars. Categorical features are also important because cars should be similar in type and configuration. Brand similarity has a smaller weight because the same brand can be useful, but it should not dominate the recommendation.

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances

pd.set_option("display.max_columns", None)


In [2]:
DATA_PATH = "cars_cleaned.csv"

cars = pd.read_csv(DATA_PATH)
cars = cars.reset_index(drop=True)
cars["car_id"] = cars.index

print(cars.shape)
cars.head()


(201, 27)


,symboling,normalized_losses,make,fuel_type,aspiration,num_doors,body_style,drive_wheels,engine_location,wheel_base,length,width,height,curb_weight,engine_type,num_cylinders,engine_size,fuel_system,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg,price,car_id
0,3,115.0,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0,0
1,3,115.0,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0,1
2,1,115.0,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154.0,5000.0,19,26,16500.0,2
3,2,164.0,audi,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.40,10.0,102.0,5500.0,24,30,13950.0,3
4,2,164.0,audi,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.40,8.0,115.0,5500.0,18,22,17450.0,4


In [3]:
cars.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   symboling          201 non-null    int64  
 1   normalized_losses  201 non-null    float64
 2   make               201 non-null    object 
 3   fuel_type          201 non-null    object 
 4   aspiration         201 non-null    object 
 5   num_doors          201 non-null    object 
 6   body_style         201 non-null    object 
 7   drive_wheels       201 non-null    object 
 8   engine_location    201 non-null    object 
 9   wheel_base         201 non-null    float64
 10  length             201 non-null    float64
 11  width              201 non-null    float64
 12  height             201 non-null    float64
 13  curb_weight        201 non-null    int64  
 14  engine_type        201 non-null    object 
 15  num_cylinders      201 non-null    object 
 16  engine_size        201 non

In [4]:
cars.describe().T


,count,mean,std,min,25%,50%,75%,max
symboling,201.0,0.840796,1.254802,-2.00,0.00,1.00,2.00,3.00
normalized_losses,201.0,120.711443,32.111623,65.00,101.00,115.00,137.00,256.00
wheel_base,201.0,98.797015,6.066366,86.60,94.50,97.00,102.40,120.90
length,201.0,174.200995,12.322175,141.10,166.80,173.20,183.50,208.10
width,201.0,65.889055,2.101471,60.30,64.10,65.50,66.60,72.00
height,201.0,53.766667,2.447822,47.80,52.00,54.10,55.50,59.80
curb_weight,201.0,2555.666667,517.296727,1488.00,2169.00,2414.00,2926.00,4066.00
engine_size,201.0,126.875622,41.546834,61.00,98.00,120.00,141.00,326.00
bore,201.0,3.330299,0.268088,2.54,3.15,3.31,3.58,3.94
stroke,201.0,3.257562,0.316082,2.07,3.11,3.29,3.41,4.17


## Feature engineering

The original dataset already contains useful features such as horsepower, engine size, MPG and price.  
To make the recommendation system stronger, we create additional features that better describe a car.


In [5]:
def add_engineered_features(cars: pd.DataFrame) -> pd.DataFrame:
    cars = cars.copy()

    # Fuel / economy features
    cars["avg_mpg"] = (cars["city_mpg"] + cars["highway_mpg"]) / 2
    cars["fuel_consumption_l_100km"] = 235.214583 / cars["avg_mpg"]

    # Performance features
    cars["power_to_weight"] = cars["horsepower"] / cars["curb_weight"] * 1000
    cars["price_per_hp"] = cars["price"] / cars["horsepower"].replace(0, np.nan)
    cars["engine_power_density"] = cars["horsepower"] / cars["engine_size"]

    # Size feature
    cars["vehicle_volume"] = cars["length"] * cars["width"] * cars["height"]

    # Category-like features generated from numerical columns
    cars["price_range"] = pd.qcut(
        cars["price"],
        q=4,
        labels=["budget", "mid", "premium", "luxury"],
        duplicates="drop",
    )
    cars["power_range"] = pd.qcut(
        cars["horsepower"],
        q=4,
        labels=["low_power", "medium_power", "high_power", "sport_power"],
        duplicates="drop",
    )
    cars["efficiency_range"] = pd.qcut(
        cars["avg_mpg"],
        q=4,
        labels=["low_efficiency", "normal_efficiency", "good_efficiency", "high_efficiency"],
        duplicates="drop",
    )
    cars["size_range"] = pd.qcut(
        cars["vehicle_volume"],
        q=4,
        labels=["compact", "medium", "large", "extra_large"],
        duplicates="drop",
    )

    # The dataset has no model name, so we create a readable label for each row
    cars["car_name"] = (
        cars["make"].str.title()
        + " "
        + cars["body_style"].str.title()
        + " "
        + cars["fuel_type"].str.title()
        + " | "
        + cars["horsepower"].round(0).astype(int).astype(str)
        + " hp | $"
        + cars["price"].round(0).astype(int).astype(str)
    )

    return cars


cars = add_engineered_features(cars)
cars[["car_id", "car_name", "avg_mpg", "fuel_consumption_l_100km", "power_to_weight", "price_range"]].head()


,car_id,car_name,avg_mpg,fuel_consumption_l_100km,power_to_weight,price_range
0,0,Alfa-Romero Convertible Gas | 111 hp | $13495,24.0,9.800608,43.563579,premium
1,1,Alfa-Romero Convertible Gas | 111 hp | $16500,24.0,9.800608,43.563579,premium
2,2,Alfa-Romero Hatchback Gas | 154 hp | $16500,22.5,10.453981,54.551895,premium
3,3,Audi Sedan Gas | 102 hp | $13950,27.0,8.711651,43.645700,premium
4,4,Audi Sedan Gas | 115 hp | $17450,20.0,11.760729,40.722380,luxury


## Feature groups

We split features into groups because different feature types should not be treated in exactly the same way.


In [6]:
NUMERIC_FEATURES = [
    "horsepower",
    "engine_size",
    "avg_mpg",
    "fuel_consumption_l_100km",
    "price",
    "curb_weight",
    "wheel_base",
    "length",
    "width",
    "height",
    "compression_ratio",
    "peak_rpm",
    "power_to_weight",
    "price_per_hp",
    "vehicle_volume",
    "engine_power_density",
]

TECHNICAL_CATEGORICAL_FEATURES = [
    "fuel_type",
    "aspiration",
    "num_doors",
    "body_style",
    "drive_wheels",
    "engine_location",
    "engine_type",
    "num_cylinders",
    "fuel_system",
    "price_range",
    "power_range",
    "efficiency_range",
    "size_range",
]

BRAND_FEATURE = ["make"]


In [7]:
def make_one_hot_encoder():
    # Compatible with both newer and older scikit-learn versions
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_similarity_matrix(
    cars: pd.DataFrame,
    numeric_weight: float = 0.55,
    categorical_weight: float = 0.35,
    brand_weight: float = 0.10,
) -> np.ndarray:
    # 1. Numerical similarity
    # RobustScaler is used because price, horsepower and engine size can contain outliers.
    numeric_data = cars[NUMERIC_FEATURES].copy()
    numeric_data = numeric_data.fillna(numeric_data.median(numeric_only=True))

    numeric_scaled = RobustScaler().fit_transform(numeric_data)

    # Convert distance to similarity using an RBF kernel.
    numeric_distances = pairwise_distances(numeric_scaled, metric="euclidean")
    sigma = np.median(numeric_distances[numeric_distances > 0])
    numeric_similarity = np.exp(-(numeric_distances ** 2) / (2 * sigma ** 2))

    # 2. Technical categorical similarity
    categorical_encoder = make_one_hot_encoder()
    categorical_matrix = categorical_encoder.fit_transform(
        cars[TECHNICAL_CATEGORICAL_FEATURES].astype(str)
    )
    categorical_similarity = cosine_similarity(categorical_matrix)

    # 3. Brand similarity
    # Brand is useful, but it should not dominate the recommendation.
    brand_encoder = make_one_hot_encoder()
    brand_matrix = brand_encoder.fit_transform(cars[BRAND_FEATURE].astype(str))
    brand_similarity = cosine_similarity(brand_matrix)

    # 4. Weighted hybrid similarity
    final_similarity = (
        numeric_weight * numeric_similarity
        + categorical_weight * categorical_similarity
        + brand_weight * brand_similarity
    )

    np.fill_diagonal(final_similarity, 1.0)
    return final_similarity


similarity_matrix = build_similarity_matrix(cars)
similarity_matrix.shape


(201, 201)

## Show available cars

Because this dataset does not contain exact model names, we use `car_id` as the input identifier.


In [8]:
def show_available_cars(cars: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    return cars[
        [
            "car_id",
            "car_name",
            "make",
            "body_style",
            "fuel_type",
            "horsepower",
            "engine_size",
            "city_mpg",
            "highway_mpg",
            "price",
        ]
    ].head(n)


show_available_cars(cars, n=15)


,car_id,car_name,make,body_style,fuel_type,horsepower,engine_size,city_mpg,highway_mpg,price
0,0,Alfa-Romero Convertible Gas | 111 hp | $13495,alfa-romero,convertible,gas,111.0,130,21,27,13495.0
1,1,Alfa-Romero Convertible Gas | 111 hp | $16500,alfa-romero,convertible,gas,111.0,130,21,27,16500.0
2,2,Alfa-Romero Hatchback Gas | 154 hp | $16500,alfa-romero,hatchback,gas,154.0,152,19,26,16500.0
3,3,Audi Sedan Gas | 102 hp | $13950,audi,sedan,gas,102.0,109,24,30,13950.0
4,4,Audi Sedan Gas | 115 hp | $17450,audi,sedan,gas,115.0,136,18,22,17450.0
5,5,Audi Sedan Gas | 110 hp | $15250,audi,sedan,gas,110.0,136,19,25,15250.0
6,6,Audi Sedan Gas | 110 hp | $17710,audi,sedan,gas,110.0,136,19,25,17710.0
7,7,Audi Wagon Gas | 110 hp | $18920,audi,wagon,gas,110.0,136,19,25,18920.0
8,8,Audi Sedan Gas | 140 hp | $23875,audi,sedan,gas,140.0,131,17,20,23875.0
9,9,Bmw Sedan Gas | 101 hp | $16430,bmw,sedan,gas,101.0,108,23,29,16430.0


## Recommendation function

The function below takes one selected car and returns a ranked list of the most similar cars.


In [9]:
def recommend_cars(
    cars: pd.DataFrame,
    similarity_matrix: np.ndarray,
    car_id: int,
    top_n: int = 10,
    include_same_make: bool = True,
) -> pd.DataFrame:
    if car_id not in cars["car_id"].values:
        raise ValueError(f"car_id={car_id} does not exist in this dataset.")

    scores = list(enumerate(similarity_matrix[car_id]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    input_car = cars.loc[car_id]
    recommendations = []

    for idx, score in scores:
        if idx == car_id:
            continue

        candidate = cars.loc[idx]

        if not include_same_make and candidate["make"] == input_car["make"]:
            continue

        recommendations.append(
            {
                "rank": len(recommendations) + 1,
                "car_id": idx,
                "similarity": round(float(score), 4),
                "car_name": candidate["car_name"],
                "make": candidate["make"],
                "body_style": candidate["body_style"],
                "fuel_type": candidate["fuel_type"],
                "drive_wheels": candidate["drive_wheels"],
                "horsepower": candidate["horsepower"],
                "engine_size": candidate["engine_size"],
                "avg_mpg": round(candidate["avg_mpg"], 1),
                "price": candidate["price"],
                "price_range": candidate["price_range"],
                "power_range": candidate["power_range"],
                "efficiency_range": candidate["efficiency_range"],
            }
        )

        if len(recommendations) == top_n:
            break

    return pd.DataFrame(recommendations)


In [10]:
selected_car_id = 3

print("Input car:")
display(cars.loc[[selected_car_id], ["car_id", "car_name", "make", "body_style", "horsepower", "engine_size", "avg_mpg", "price"]])

print("Recommended similar cars:")
recommendations = recommend_cars(cars, similarity_matrix, selected_car_id, top_n=10)
recommendations


Input car:


,car_id,car_name,make,body_style,horsepower,engine_size,avg_mpg,price
3,3,Audi Sedan Gas | 102 hp | $13950,audi,sedan,102.0,109,27.0,13950.0


Recommended similar cars:


,rank,car_id,similarity,car_name,make,body_style,fuel_type,drive_wheels,horsepower,engine_size,avg_mpg,price,price_range,power_range,efficiency_range
0,1,40,0.8658,Honda Sedan Gas | 101 hp | $12945,honda,sedan,gas,fwd,101.0,110,26.0,12945.0,premium,high_power,normal_efficiency
1,2,5,0.8414,Audi Sedan Gas | 110 hp | $15250,audi,sedan,gas,fwd,110.0,136,22.0,15250.0,premium,high_power,low_efficiency
2,3,131,0.8340,Saab Sedan Gas | 110 hp | $15510,saab,sedan,gas,fwd,110.0,121,24.5,15510.0,premium,high_power,normal_efficiency
3,4,129,0.8310,Saab Sedan Gas | 110 hp | $12170,saab,sedan,gas,fwd,110.0,121,24.5,12170.0,premium,high_power,normal_efficiency
4,5,184,0.8244,Volkswagen Sedan Gas | 100 hp | $9995,volkswagen,sedan,gas,fwd,100.0,109,29.0,9995.0,mid,high_power,good_efficiency
5,6,9,0.7981,Bmw Sedan Gas | 101 hp | $16430,bmw,sedan,gas,rwd,101.0,108,26.0,16430.0,premium,high_power,normal_efficiency
6,7,10,0.7955,Bmw Sedan Gas | 101 hp | $16925,bmw,sedan,gas,rwd,101.0,108,26.0,16925.0,luxury,high_power,normal_efficiency
7,8,130,0.7822,Saab Hatchback Gas | 110 hp | $15040,saab,hatchback,gas,fwd,110.0,121,24.5,15040.0,premium,high_power,normal_efficiency
8,9,192,0.7820,Volvo Sedan Gas | 114 hp | $15985,volvo,sedan,gas,rwd,114.0,141,26.0,15985.0,premium,high_power,normal_efficiency
9,10,190,0.7794,Volvo Sedan Gas | 114 hp | $12940,volvo,sedan,gas,rwd,114.0,141,25.5,12940.0,premium,high_power,normal_efficiency


## Explain one recommendation

This makes the system more transparent: we can show why a car was recommended.


In [11]:
def explain_recommendation(cars: pd.DataFrame, input_car_id: int, recommended_car_id: int) -> pd.DataFrame:
    compare_features = [
        "make",
        "body_style",
        "fuel_type",
        "aspiration",
        "drive_wheels",
        "horsepower",
        "engine_size",
        "avg_mpg",
        "fuel_consumption_l_100km",
        "curb_weight",
        "price",
        "price_range",
        "power_range",
        "efficiency_range",
    ]

    input_car = cars.loc[input_car_id, compare_features]
    recommended_car = cars.loc[recommended_car_id, compare_features]

    explanation = pd.DataFrame(
        {
            "feature": compare_features,
            "input_car": input_car.values,
            "recommended_car": recommended_car.values,
        }
    )

    explanation["same_or_difference"] = np.where(
        explanation["input_car"].astype(str) == explanation["recommended_car"].astype(str),
        "same",
        "different",
    )

    return explanation


first_recommended_car_id = int(recommendations.iloc[0]["car_id"])
explain_recommendation(cars, selected_car_id, first_recommended_car_id)


,feature,input_car,recommended_car,same_or_difference
0,make,audi,honda,different
1,body_style,sedan,sedan,same
2,fuel_type,gas,gas,same
3,aspiration,std,std,same
4,drive_wheels,fwd,fwd,same
5,horsepower,102.0,101.0,different
6,engine_size,109,110,different
7,avg_mpg,27.0,26.0,different
8,fuel_consumption_l_100km,8.711651,9.046715,different
9,curb_weight,2337,2465,different


## Sanity-check evaluation

This dataset has no user ratings, so we cannot calculate RMSE or precision@k against real user behaviour.  
Instead, we perform a content-based sanity check: for each car, we check whether top recommendations share important properties.


In [12]:
def evaluate_recommendations(cars: pd.DataFrame, similarity_matrix: np.ndarray, top_n: int = 5) -> pd.DataFrame:
    body_matches = []
    fuel_matches = []
    drive_matches = []
    price_range_matches = []

    for car_id in cars["car_id"]:
        recs = recommend_cars(cars, similarity_matrix, car_id, top_n=top_n)
        input_car = cars.loc[car_id]

        body_matches.append((recs["body_style"] == input_car["body_style"]).mean())
        fuel_matches.append((recs["fuel_type"] == input_car["fuel_type"]).mean())
        drive_matches.append((recs["drive_wheels"] == input_car["drive_wheels"]).mean())
        price_range_matches.append((recs["price_range"].astype(str) == str(input_car["price_range"])).mean())

    return pd.DataFrame(
        {
            "metric": [
                "same body_style rate",
                "same fuel_type rate",
                "same drive_wheels rate",
                "same price_range rate",
            ],
            f"average_in_top_{top_n}": [
                round(float(np.mean(body_matches)), 3),
                round(float(np.mean(fuel_matches)), 3),
                round(float(np.mean(drive_matches)), 3),
                round(float(np.mean(price_range_matches)), 3),
            ],
        }
    )


evaluate_recommendations(cars, similarity_matrix, top_n=5)


,metric,average_in_top_5
0,same body_style rate,0.589
1,same fuel_type rate,1.000
2,same drive_wheels rate,0.883
3,same price_range rate,0.702
